# 🤖 Notebook 4: Modelos Clásicos de Machine Learning

## 🎯 Objetivos de este Notebook

En este notebook aprenderás:
1. ✅ Entrenar **Naive Bayes** para clasificación de texto
2. ✅ Entrenar **SVM** (Support Vector Machine)
3. ✅ **Evaluar modelos** con métricas (accuracy, precision, recall)
4. ✅ Ver **palabras más importantes** para cada sentimiento
5. ✅ **Comparar** ambos modelos

⏱️ **Tiempo estimado**: 30 minutos

---

## 🔧 Setup Inicial

In [ ]:
# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN DE RUTAS - Solución robusta para Jupyter
# ============================================================================

def find_project_root():
    """Encuentra la raíz del proyecto buscando config.py y src/"""
    current = Path.cwd()
    
    # Buscar hacia arriba hasta 5 niveles
    for _ in range(5):
        config_exists = (current / 'config.py').exists()
        src_exists = (current / 'src').exists()
        
        if config_exists and src_exists:
            return current
        
        # También verificar si estamos dentro de sentiment-analysis
        if current.name == 'sentiment-analysis' and src_exists:
            return current
            
        current = current.parent
    
    # Si no encuentra, asumir que es el directorio actual
    return Path.cwd()

# Encontrar y configurar la ruta del proyecto
project_root = find_project_root()
print(f"📁 Proyecto encontrado en: {project_root}")

# Agregar al path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Cambiar el working directory
os.chdir(str(project_root))

# Importar módulos
from src import data_loader, text_preprocessing, feature_extraction
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

print("✅ Setup completo!")

## 📥 Preparar Datos

Vamos a usar un subset del dataset IMDB para que el entrenamiento sea más rápido:

In [ ]:
# Cargar dataset IMDB
print("📥 Cargando dataset IMDB...")
(X_train_raw, y_train), (X_test_raw, y_test) = data_loader.load_imdb_data()

# Usar subset para demo (más rápido)
n_samples = 5000  # Usa 5000 reviews de cada conjunto
X_train_raw = X_train_raw[:n_samples]
y_train = y_train[:n_samples]
X_test_raw = X_test_raw[:n_samples]
y_test = y_test[:n_samples]

print(f"\n✅ Datos cargados:")
print(f"   • Training: {len(X_train_raw)} reviews")
print(f"   • Test: {len(X_test_raw)} reviews")

## 🔄 Decodificar y Preprocesar

Convertir números → texto → texto limpio:

In [ ]:
# Obtener diccionario de palabras
print("📖 Cargando diccionario...")
word_index = data_loader.get_word_index()

# Decodificar reviews (números → texto)
print("\n🔄 Decodificando reviews...")
train_texts = [data_loader.decode_review(review, word_index) for review in X_train_raw]
test_texts = [data_loader.decode_review(review, word_index) for review in X_test_raw]

print("✅ Reviews decodificadas")

# Preprocesar (limpiar texto)
print("\n🧹 Preprocesando texto...")
train_texts_clean = [text_preprocessing.preprocess_text(text) for text in train_texts]
test_texts_clean = [text_preprocessing.preprocess_text(text) for text in test_texts]

print("✅ Texto preprocesado")

# Ver ejemplo
print("\n📝 Ejemplo de preprocesamiento:")
print(f"Original:  '{train_texts[0][:100]}...'")
print(f"Limpio:    '{train_texts_clean[0][:100]}...'")

## 🔢 Crear Features TF-IDF

In [ ]:
# Crear TF-IDF features
print("📊 Creando features TF-IDF...")
X_train_tfidf, vectorizer = feature_extraction.create_tfidf_features(
    train_texts_clean,
    max_features=5000  # Top 5000 palabras
)

# Transformar test set con el MISMO vectorizer
X_test_tfidf, _ = feature_extraction.create_tfidf_features(
    test_texts_clean,
    vectorizer=vectorizer  # Importante: usar el mismo!
)

print(f"\n✅ Features creadas:")
print(f"   • Training shape: {X_train_tfidf.shape}")
print(f"   • Test shape: {X_test_tfidf.shape}")
print(f"   • Vocabulario: {len(vectorizer.vocabulary_)} palabras")

## 🧮 Modelo 1: Naive Bayes

### 💡 ¿Cómo funciona Naive Bayes?

Calcula **probabilidades**:

```
P(positivo | "excellent movie") = ?

Usando Teorema de Bayes:
P(positivo | palabras) = P(palabras | positivo) × P(positivo) / P(palabras)
```

### Entrenar:

In [ ]:
# Crear y entrenar modelo
print("🧮 Entrenando Naive Bayes...")
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)

print("\n✅ Naive Bayes entrenado!")

# Predecir
y_pred_nb = nb_model.predict(X_test_tfidf)

# Calcular métricas
acc_nb = accuracy_score(y_test, y_pred_nb)
prec_nb = precision_score(y_test, y_pred_nb)
rec_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

print("\n📊 RESULTADOS NAIVE BAYES:")
print("="*50)
print(f"   • Accuracy:  {acc_nb:.4f} ({acc_nb*100:.2f}%)")
print(f"   • Precision: {prec_nb:.4f} ({prec_nb*100:.2f}%)")
print(f"   • Recall:    {rec_nb:.4f} ({rec_nb*100:.2f}%)")
print(f"   • F1-Score:  {f1_nb:.4f}")
print("="*50)

## 🎯 Modelo 2: SVM (Support Vector Machine)

### 💡 ¿Cómo funciona SVM?

Encuentra un **hiperplano** (línea en alta dimensión) que separa positivos de negativos con el **máximo margen**.

### Entrenar:

In [ ]:
# Crear y entrenar modelo
print("🎯 Entrenando SVM...")
svm_model = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svm_model.fit(X_train_tfidf, y_train)

print("\n✅ SVM entrenado!")

# Predecir
y_pred_svm = svm_model.predict(X_test_tfidf)

# Calcular métricas
acc_svm = accuracy_score(y_test, y_pred_svm)
prec_svm = precision_score(y_test, y_pred_svm)
rec_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

print("\n📊 RESULTADOS SVM:")
print("="*50)
print(f"   • Accuracy:  {acc_svm:.4f} ({acc_svm*100:.2f}%)")
print(f"   • Precision: {prec_svm:.4f} ({prec_svm*100:.2f}%)")
print(f"   • Recall:    {rec_svm:.4f} ({rec_svm*100:.2f}%)")
print(f"   • F1-Score:  {f1_svm:.4f}")
print("="*50)

## 📊 Comparación de Modelos

In [ ]:
# Crear DataFrame comparativo
comparacion = pd.DataFrame({
    'Modelo': ['Naive Bayes', 'SVM'],
    'Accuracy': [acc_nb, acc_svm],
    'Precision': [prec_nb, prec_svm],
    'Recall': [rec_nb, rec_svm],
    'F1-Score': [f1_nb, f1_svm]
})

print("\n🏆 COMPARACIÓN DE MODELOS:")
print("="*70)
print(comparacion.to_string(index=False))
print("="*70)

# Visualizar comparación
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparacion))
width = 0.2

ax.bar(x - 1.5*width, comparacion['Accuracy'], width, label='Accuracy', alpha=0.8)
ax.bar(x - 0.5*width, comparacion['Precision'], width, label='Precision', alpha=0.8)
ax.bar(x + 0.5*width, comparacion['Recall'], width, label='Recall', alpha=0.8)
ax.bar(x + 1.5*width, comparacion['F1-Score'], width, label='F1-Score', alpha=0.8)

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparación de Métricas: Naive Bayes vs SVM', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparacion['Modelo'])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Ganador
mejor_modelo = 'SVM' if acc_svm > acc_nb else 'Naive Bayes'
print(f"\n🏆 Mejor modelo: {mejor_modelo}")

## 🔍 Análisis de Feature Importance (SVM)

¿Qué palabras son más importantes para cada sentimiento?

In [ ]:
# Obtener coeficientes del modelo SVM
coef = svm_model.coef_[0]
feature_names = vectorizer.get_feature_names_out()

# Top 20 palabras POSITIVAS (coeficiente más alto)
top_positive_idx = np.argsort(coef)[-20:][::-1]
top_positive_words = [(feature_names[i], coef[i]) for i in top_positive_idx]

# Top 20 palabras NEGATIVAS (coeficiente más bajo)
top_negative_idx = np.argsort(coef)[:20]
top_negative_words = [(feature_names[i], coef[i]) for i in top_negative_idx]

print("\n🟢 TOP 20 PALABRAS MÁS POSITIVAS:")
print("="*50)
for i, (word, score) in enumerate(top_positive_words, 1):
    print(f"   {i:2}. '{word}': {score:.4f}")

print("\n🔴 TOP 20 PALABRAS MÁS NEGATIVAS:")
print("="*50)
for i, (word, score) in enumerate(top_negative_words, 1):
    print(f"   {i:2}. '{word}': {score:.4f}")

## 📈 Visualizar Feature Importance

In [ ]:
# Crear visualización
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Palabras positivas
pos_words, pos_scores = zip(*top_positive_words)
axes[0].barh(range(len(pos_words)), pos_scores, color='green', alpha=0.7)
axes[0].set_yticks(range(len(pos_words)))
axes[0].set_yticklabels(pos_words)
axes[0].set_xlabel('Coeficiente SVM', fontsize=12)
axes[0].set_title('🟢 Top 20 Palabras Positivas', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

# Palabras negativas
neg_words, neg_scores = zip(*top_negative_words)
axes[1].barh(range(len(neg_words)), neg_scores, color='red', alpha=0.7)
axes[1].set_yticks(range(len(neg_words)))
axes[1].set_yticklabels(neg_words)
axes[1].set_xlabel('Coeficiente SVM', fontsize=12)
axes[1].set_title('🔴 Top 20 Palabras Negativas', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 Resumen

En este notebook aprendiste:

✅ **Naive Bayes**: Modelo probabilístico rápido
✅ **SVM**: Encuentra hiperplano óptimo
✅ **Métricas**: Accuracy, Precision, Recall, F1
✅ **Feature Importance**: Qué palabras importan más
✅ **Comparación**: SVM generalmente supera a Naive Bayes

---

## 🎓 Próximo Paso

En el **Notebook 5** aprenderás:
- 🧠 Word Embeddings y LSTM
- 🔄 Por qué LSTM entiende "not good"
- 🏗️ Construir red neuronal para NLP

**¡Nos vemos en el siguiente notebook!** 🚀